In [ ]:
%pip install -q transformers datasets accelerate evaluate sentencepiece torch pandas

In [ ]:
import torch
import json
import pandas as pd
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, TrainingArguments, Trainer, LlamaTokenizerFast

In [ ]:
MODEL_PRETRAINED = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"       
MODEL_FINETUNED_PATH = "r1-qg-squad-finetuned"    
LOCAL_JSON_FILE_PATH = "train-v2.0.json"
TRAIN_SAMPLES = 1000                            
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
with open(LOCAL_JSON_FILE_PATH, 'r') as f:
    squad_data = json.load(f)['data']

processed_examples = []
for topic in squad_data:
    for paragraph in topic['paragraphs']:
        context = paragraph['context']
        for qas in paragraph['qas']:
            question = qas['question']
            is_impossible = qas['is_impossible']

            processed_examples.append({
                'context': context,
                'question': question,
                'is_impossible': is_impossible
            })

df = pd.DataFrame(processed_examples)
raw_dataset = Dataset.from_pandas(df)


valid_dataset = raw_dataset.filter(lambda example: not example['is_impossible'])
train_dataset = valid_dataset.select(range(TRAIN_SAMPLES))

df.head()

Filter:   0%|          | 0/130319 [00:00<?, ? examples/s]

Successfully loaded and processed the data.
Total valid examples found: 86821
Using 1000 examples for training.
Done.


,context,question,is_impossible
0,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,When did Beyonce start becoming popular?,False
1,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,What areas did Beyonce compete in when she was...,False
2,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,When did Beyonce leave Destiny's Child and bec...,False
3,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,In what city and state did Beyonce grow up?,False
4,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,In which decade did Beyonce become famous?,False


In [ ]:
tokenizer = LlamaTokenizerFast.from_pretrained(MODEL_PRETRAINED)

def preprocess_for_qg(examples):
    """Formats and tokenizes the text data for the T5 model."""
    inputs = [f"generate question: {context}" for context in examples['context']]
    targets = [question for question in examples['question']]

    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_dataset = train_dataset.map(preprocess_for_qg, batched=True, remove_columns=train_dataset.column_names)

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Done.


In [ ]:
model = T5ForConditionalGeneration.from_pretrained(MODEL_PRETRAINED).to(DEVICE)

training_args = TrainingArguments(
    output_dir=MODEL_FINETUNED_PATH,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=500,
    save_steps=1000,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

trainer.save_model()

NameError: name 'T5ForConditionalGeneration' is not defined

In [ ]:
fine_tuned_model = T5ForConditionalGeneration.from_pretrained(MODEL_FINETUNED_PATH).to(DEVICE)
fine_tuned_tokenizer = T5Tokenizer.from_pretrained(MODEL_FINETUNED_PATH)

context = """
A large language model (LLM) is a type of artificial intelligence (AI) program that can recognize and generate text, among other tasks. LLMs are trained on huge sets of data — hence the name "large." LLMs are built on machine learning: specifically, a type of neural network called a transformer model.
In simpler terms, an LLM is a computer program that has been fed enough examples to be able to recognize and interpret human language or other types of complex data. Many LLMs are trained on data that has been gathered from the Internet — thousands or millions of gigabytes' worth of text. But the quality of the samples impacts how well LLMs will learn natural language, so an LLM's programmers may use a more curated data set.
LLMs use a type of machine learning called deep learning in order to understand how characters, words, and sentences function together. Deep learning involves the probabilistic analysis of unstructured data, which eventually enables the deep learning model to recognize distinctions between pieces of content without human intervention.
LLMs are then further trained via tuning: they are fine-tuned or prompt-tuned to the particular task that the programmer wants them to do, such as interpreting questions and generating responses, or translating text from one language to another.
"""

input_text = f"generate question: {context}"
inputs = fine_tuned_tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(DEVICE)


output_sequences = fine_tuned_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=100,
    num_beams=5,  
    early_stopping=True,
    num_return_sequences=3,
)

generated_question = fine_tuned_tokenizer.decode(output_sequences[0], skip_special_tokens=True)

print(f"CONTEXT:\n{context}")

print("\nGENERATED QUESTIONS:")
for i, sequence in enumerate(output_sequences):
    generated_question = fine_tuned_tokenizer.decode(sequence, skip_special_tokens=True)
    print(f"Question {i+1}: {generated_question}")


--- 6. Testing the new fine-tuned model... ---
CONTEXT:

A large language model (LLM) is a type of artificial intelligence (AI) program that can recognize and generate text, among other tasks. LLMs are trained on huge sets of data — hence the name "large." LLMs are built on machine learning: specifically, a type of neural network called a transformer model.

In simpler terms, an LLM is a computer program that has been fed enough examples to be able to recognize and interpret human language or other types of complex data. Many LLMs are trained on data that has been gathered from the Internet — thousands or millions of gigabytes' worth of text. But the quality of the samples impacts how well LLMs will learn natural language, so an LLM's programmers may use a more curated data set.

LLMs use a type of machine learning called deep learning in order to understand how characters, words, and sentences function together. Deep learning involves the probabilistic analysis of unstructured data, 